<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">End-to-End ML: Titanic Survival Prediction</h1></center>

A complete machine learning workflow from raw data to a final model.

### Workflow:
1. [Exploratory Data Analysis](#1)
2. [Feature Engineering](#2)
3. [Baseline & Model Comparison](#3)
4. [Hyperparameter Tuning](#4)
5. [Threshold Tuning](#5)
6. [Final Evaluation](#6)
7. [Feature Importance](#7)

In [ ]:
import seaborn as sns
titanic_colors = ['#14213d', '#fca311', '#e5e5e5', '#ffffff', '#000000']
print("Titanic Colors:")
sns.palplot(sns.color_palette(titanic_colors[:4]))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, log_loss,
                              classification_report, confusion_matrix,
                              RocCurveDisplay, PrecisionRecallDisplay)
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Exploratory Data Analysis</h1></center>

<center><h1 style = "background:#14213d ;color:white;border:0;font-weight:bold">Information About Dataset</h1></center>

**Titanic** — 891 passengers, binary target: survived (1) or died (0).

| Feature | Description |
|---|---|
| pclass | Passenger class (1=First, 2=Second, 3=Third) |
| sex | Gender |
| age | Age in years |
| sibsp | # siblings/spouses aboard |
| parch | # parents/children aboard |
| fare | Ticket fare |
| embarked | Port of embarkation (C/Q/S) |
| **survived** | **Target: 1 = survived** |

In [ ]:
try:
    df = sns.load_dataset("titanic")
except Exception:
    from sklearn.datasets import fetch_openml
    raw = fetch_openml("titanic", version=1, as_frame=True)
    df = raw.frame
    df["survived"] = df["survived"].astype(int)

print(f"Shape: {df.shape}")
print(f"Survival rate: {df['survived'].mean():.3f}")
df.head()

In [ ]:
missing = df.isnull().sum()
mv = pd.DataFrame({"Missing": missing, "Pct": (missing/len(df)*100).round(1)})
mv[mv["Missing"] > 0].sort_values("Missing", ascending=False)

<center><h1 style = "background:#fca311 ;color:black;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9), dpi=100)

axes[0,0].bar(["Died","Survived"],
              df["survived"].value_counts().sort_index().values,
              color=["#14213d","#fca311"])
axes[0,0].set_title("Survival Count")

surv_sex = df.groupby("sex")["survived"].mean()
axes[0,1].bar(surv_sex.index, surv_sex.values, color=["#fca311","#14213d"])
axes[0,1].set_title("Survival Rate by Sex")
axes[0,1].set_ylabel("Rate")

surv_cls = df.groupby("pclass")["survived"].mean()
axes[0,2].bar(surv_cls.index.astype(str), surv_cls.values, color=["gold","silver","peru"])
axes[0,2].set_title("Survival Rate by Class")

df[df["survived"]==1]["age"].dropna().hist(ax=axes[1,0], bins=25, alpha=0.6,
                                            label="Survived", color="#fca311")
df[df["survived"]==0]["age"].dropna().hist(ax=axes[1,0], bins=25, alpha=0.6,
                                            label="Died", color="#14213d")
axes[1,0].legend(); axes[1,0].set_title("Age by Survival")

df["family_size"] = df["sibsp"] + df["parch"] + 1
surv_fam = df.groupby("family_size")["survived"].mean()
axes[1,1].bar(surv_fam.index.astype(str), surv_fam.values, color="#14213d")
axes[1,1].set_title("Survival by Family Size")
axes[1,1].set_xlabel("Family size")

df["log_fare"] = np.log1p(df["fare"])
sns.kdeplot(df[df["survived"]==1]["log_fare"], ax=axes[1,2], fill=True,
            label="Survived", color="#fca311", alpha=0.6)
sns.kdeplot(df[df["survived"]==0]["log_fare"], ax=axes[1,2], fill=True,
            label="Died", color="#14213d", alpha=0.6)
axes[1,2].legend(); axes[1,2].set_title("Fare (log) by Survival")

plt.tight_layout(); plt.show()

<a id = "2"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Feature Engineering</h1></center>

<center><h1 style = "background:#14213d ;color:white;border:0;font-weight:bold">Train-Test Split</h1></center>

In [ ]:
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"]    = (df["family_size"] == 1).astype(int)
df["log_fare"]    = np.log1p(df["fare"])

num_features = ["age", "log_fare", "family_size", "is_alone"]
cat_features = ["sex", "pclass", "embark_town"]

df2 = df.dropna(subset=["survived"])
X = df2[num_features + cat_features]
y = df2["survived"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features)
])

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Positive rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

<a id = "3"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Baseline & Model Comparison</h1></center>

<center><h1 style = "background:#fca311 ;color:black;border:0;font-weight:bold">Model Comparison</h1></center>

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Baseline (majority)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
    "KNN":                 KNeighborsClassifier(n_neighbors=9),
}

cv_results = []
for name, clf in models.items():
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    accs = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="accuracy")
    aucs = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="roc_auc")
    cv_results.append({"Model": name, "CV Acc": accs.mean().round(4),
                        "Acc Std": accs.std().round(4),
                        "CV AUC": aucs.mean().round(4),
                        "AUC Std": aucs.std().round(4)})

res_df = pd.DataFrame(cv_results).sort_values("CV AUC", ascending=False)
res_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), dpi=100)
x = np.arange(len(res_df)); w = 0.35
ax.bar(x-w/2, res_df["CV Acc"], w, label="CV Accuracy", color="#14213d",
       yerr=res_df["Acc Std"], capsize=4)
ax.bar(x+w/2, res_df["CV AUC"], w, label="CV ROC-AUC", color="#fca311",
       yerr=res_df["AUC Std"], capsize=4)
ax.set_xticks(x); ax.set_xticklabels(res_df["Model"], rotation=20, ha="right")
ax.set_ylabel("Score"); ax.set_ylim(0.5, 1.0)
ax.set_title("Model Comparison", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

<a id = "4"></a>
<center><h1 style = "background:#14213d ;color:white;border:0;font-weight:bold">Hyperparameter Tuning (Random Forest)</h1></center>

In [ ]:
param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth":    [None, 8, 12],
    "clf__min_samples_leaf": [1, 3],
    "clf__max_features": ["sqrt", "log2"],
}

rf_pipe = Pipeline([("pre", preprocessor),
                     ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))])

grid_search = GridSearchCV(rf_pipe, param_grid, cv=skf, scoring="roc_auc",
                            n_jobs=-1, refit=True)
grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_

<a id = "5"></a>
<center><h1 style = "background:#fca311 ;color:black;border:0;font-weight:bold">Threshold Tuning</h1></center>

In [ ]:
y_prob = best_model.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.1, 0.9, 81)
f1s = [f1_score(y_test, (y_prob >= t).astype(int)) for t in thresholds]
best_t = thresholds[np.argmax(f1s)]

plt.figure(figsize=(9, 4), dpi=100)
plt.plot(thresholds, f1s, color="#14213d", lw=2)
plt.axvline(best_t, color="#fca311", ls="--", label=f"Best threshold={best_t:.2f}")
plt.axvline(0.5,    color="gray",    ls=":",  label="Default=0.5")
plt.xlabel("Threshold"); plt.ylabel("F1 Score")
plt.title("F1 Score vs Classification Threshold", fontweight="bold")
plt.legend(); plt.show()
print(f"Best threshold: {best_t:.2f}  F1={max(f1s):.4f}")

<a id = "6"></a>
<center><h1 style = "background:#14213d ;color:white;border:0;font-weight:bold">Final Evaluation</h1></center>

In [ ]:
y_pred_final = (y_prob >= best_t).astype(int)

results = [accuracy_score(y_test, y_pred_final),
           precision_score(y_test, y_pred_final),
           recall_score(y_test, y_pred_final),
           f1_score(y_test, y_pred_final),
           roc_auc_score(y_test, y_prob),
           log_loss(y_test, y_prob)]
pd.DataFrame({"Metric": ["Accuracy","Precision","Recall","F1","ROC-AUC","Log-Loss"],
              "Score": results}).round(4)

In [ ]:
print(classification_report(y_test, y_pred_final, target_names=["Died","Survived"]))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=100)

cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Died","Survived"],
            yticklabels=["Died","Survived"], ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], color="#14213d")
axes[1].set_title("ROC Curve")

PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[2], color="#fca311")
axes[2].set_title("Precision-Recall Curve")

plt.tight_layout(); plt.show()

<a id = "7"></a>
<center><h1 style = "background:#fca311 ;color:black;border:0;font-weight:bold">Feature Importance</h1></center>

In [ ]:
rf_clf = best_model.named_steps["clf"]
pre    = best_model.named_steps["pre"]

try:
    cat_names = pre.named_transformers_["cat"]["ohe"].get_feature_names_out(cat_features).tolist()
except Exception:
    cat_names = []

all_names = num_features + cat_names
importances = rf_clf.feature_importances_

if len(all_names) == len(importances):
    imp_df = pd.DataFrame({"Feature": all_names, "Importance": importances})
    imp_df = imp_df.sort_values("Importance", ascending=True).tail(15)

    plt.figure(figsize=(9, 6), dpi=100)
    plt.barh(imp_df["Feature"], imp_df["Importance"],
             color="#14213d", edgecolor="white")
    plt.xlabel("Feature Importance (MDI)")
    plt.title("Top 15 Feature Importances — Titanic RF", fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("Feature name count mismatch — run with full pipeline")

## Conclusion

This end-to-end notebook demonstrated the full ML workflow:

1. **EDA** — understand distributions, missing values, and relationships
2. **Feature Engineering** — family size, is_alone, log_fare improve signal
3. **Preprocessing Pipeline** — prevents data leakage; handles imputation + encoding
4. **Model Comparison** — consistent CV comparison across 6 models
5. **Hyperparameter Tuning** — GridSearchCV finds optimal RF configuration
6. **Threshold Tuning** — default 0.5 is not always optimal
7. **Full Evaluation** — accuracy alone can mislead; always report precision, recall, F1, AUC